# Trust Analysis of Traffic Sign Classifiers - Training & Stress Suite

This notebook provides the complete pipeline for training and evaluating Traffic Sign Classifiers for the TSR Thesis project. It is optimized for Google Colab with GPU acceleration.

### Pipeline Overview:
1. **Environment Setup**: Repository cloning and dependency installation.
2. **Data Preparation**: Downloading and restructuring the GTSRB dataset.
3. **Model Training**: Running the training loop with W&B logging.
4. **Uncertainty Stress Suite**: Evaluating robustness against noise, blur, and weather corruptions.
5. **Result Export**: Automated downloading of metrics and model weights.

## 1. Environment Setup
Cloning the latest version of the repository and installing scientific dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/ucefhesham/tsr-thesis.git
%cd tsr-thesis

# Install dependencies (quietly)
!pip install -r requirements.txt -q
!pip install wandb -U -q

print("\n--- Setup Complete ---")

## 2. Weights & Biases Logging
Login is required to sync your training progress and model checkpoints.

In [ ]:
import wandb
wandb.login()

## 3. Data Preparation
GTSRB is a large dataset (~300MB). This cell ensures all partitions (Train, Test, and Test Ground Truth) are correctly downloaded into the Colab environment.

In [ ]:
from src.datamodules.gtsrb_module import GTSRBDataModule

dm = GTSRBDataModule(data_dir="data/")
dm.prepare_data()
print("\n--- GTSRB Dataset Ready ---")

## 4. Model Training
You can train the standard **ResNet-18 Baseline** or the **Evidential Model** by passing the appropriate model config.

In [ ]:
# OPTION A: Train ResNet-18 Baseline
!python train.py model=resnet18 trainer.max_epochs=20 trainer.devices=1 trainer.accelerator='gpu'

# OPTION B: Train Evidential Model (uncomment below)
# !python train.py model=evidential trainer.max_epochs=20 trainer.devices=1 trainer.accelerator='gpu'

## 5. Uncertainty Stress Suite
Evaluate robustness across 4 corruption categories and 5 severity levels. This script automatically performs temperature scaling and exports a final CSV for analysis.

In [ ]:
# Provide the path to your best checkpoint (trained in step 4 or downloaded)
CHECKPOINT_PATH = "checkpoints/best-trust-baseline.ckpt"

# Run the automated Stress Suite sweep
# Note: The script finds the GPU automatically!
!python eval.py ckpt_path={CHECKPOINT_PATH}

print("\n--- Evaluation Complete ---")

## 6. Export Results
Download the metrics CSV and the model weights to your local machine.

In [ ]:
from google.colab import files
import os

# 1. Download Stress Test Results
if os.path.exists('logs/stress_test_results.csv'):
    files.download('logs/stress_test_results.csv')

# 2. Download Model Checkpoint (optional)
# Ensure you use the correct filename generated during training
if os.path.exists('checkpoints/best-trust-baseline.ckpt'):
    files.download('checkpoints/best-trust-baseline.ckpt')